# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shiva-sn/ML/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [5]:
import pandas as pd
import numpy as np

df = pd.read_csv("/content/content_refresh_anonymized.csv")

print("Shape:", df.shape)
display(df.head())
print("Number of columns:", len(df.columns))
print("\nColumns:")
for i, col in enumerate(df.columns, start=1):
    print(f"{i}. {col}")

candidate_signals = [
    "search_volume",
    "competition",
    "cpc",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

candidate_signals = [
    col for col in candidate_signals
    if col in df.columns
]

signal_inventory = pd.DataFrame({
    "signal": candidate_signals,
    "dtype": [df[col].dtype for col in candidate_signals],
    "n": [df[col].notna().sum() for col in candidate_signals],
    "missing_n": [df[col].isna().sum() for col in candidate_signals],
    "missing_pct": [
        df[col].isna().mean() * 100
        for col in candidate_signals
    ],
    "unique_n": [
        df[col].nunique(dropna=True)
        for col in candidate_signals
    ]
})

display(signal_inventory)

Shape: (30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


Number of columns: 44

Columns:
1. content_id
2. client_id
3. search_volume
4. competition
5. competition_level
6. cpc
7. content_type
8. main_intent
9. word_count
10. char_count
11. provider_used
12. model_used
13. impressions_90d
14. clicks_90d
15. pageviews_90d
16. sessions_90d
17. users_90d
18. engaged_sessions_90d
19. ai_sessions_90d
20. scroll_events_90d
21. days_with_impressions
22. days_with_sessions
23. impressions_last_30d
24. clicks_last_30d
25. sessions_last_30d
26. impressions_prev_30d
27. clicks_prev_30d
28. sessions_prev_30d
29. content_age_days
30. age_tier
31. age_tier_order
32. days_since_last_update
33. freshness_tier
34. word_count_tier
35. char_count_tier
36. ctr
37. avg_position
38. engagement_rate
39. scroll_rate
40. ai_traffic_pct
41. impression_tier
42. position_tier
43. trend_direction
44. trend_pct


,signal,dtype,n,missing_n,missing_pct,unique_n
0,search_volume,float64,27532,2468,8.226667,41
1,competition,float64,27532,2468,8.226667,101
2,cpc,float64,27532,2468,8.226667,915
3,impressions_90d,int64,30000,0,0.000000,9438
4,clicks_90d,int64,30000,0,0.000000,477
5,pageviews_90d,int64,30000,0,0.000000,856
6,sessions_90d,int64,30000,0,0.000000,666
7,engaged_sessions_90d,int64,30000,0,0.000000,68
8,ai_sessions_90d,int64,30000,0,0.000000,35
9,scroll_events_90d,int64,30000,0,0.000000,155


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [6]:
df["days_since_last_update"].describe()

,days_since_last_update
count,30000.000000
mean,46.098300
std,42.078709
min,1.000000
25%,20.000000
50%,20.000000
75%,104.000000
max,373.000000


In [8]:
df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-1, 30, 90, 180, 365, np.inf],
    labels=[
        "0-30",
        "31-90",
        "91-180",
        "181-365",
        "365+"
    ]
)
staleness_audit = (
    df.groupby(
        "staleness_bucket",
        observed=False
    )
    .agg(
        n=("content_id", "size"),
        median_impressions_90d=("impressions_90d", "median"),
        median_clicks_90d=("clicks_90d", "median"),
        median_ctr=("ctr", "median"),
        median_position=("avg_position", "median")
    )
    .reset_index()
)

display(staleness_audit)

,staleness_bucket,n,median_impressions_90d,median_clicks_90d,median_ctr,median_position
0,0-30,20480,470.0,1.0,0.04,9.9
1,31-90,175,510.0,0.0,0.00,13.9
2,91-180,9171,1692.0,2.0,0.10,13.6
3,181-365,169,16.0,0.0,0.00,7.0
4,365+,5,2.0,0.0,0.00,7.5


In [9]:
display(
    df["staleness_bucket"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("bucket")
    .reset_index(name="n")
)

,bucket,n
0,0-30,20480
1,31-90,175
2,91-180,9171
3,181-365,169
4,365+,5


### Staleness Signal Verdict

**Verdict: MIXED**

The relationship between days since the last update and current performance is not consistently monotonic across all buckets. Therefore, staleness is not strong enough to be treated as a standalone predictor of poor performance. However, it remains a reasonable supporting signal for content-refresh prioritization.

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [10]:
df["search_volume"].describe()
print(
    "Missing:",
    df["search_volume"].isna().sum()
)

print(
    "Missing %:",
    round(
        df["search_volume"].isna().mean() * 100,
        2
    )
)

Missing: 2468
Missing %: 8.23


In [12]:
print(
    "Zero search-volume rows:",
    (df["search_volume"] == 0).sum()
)
df["volume_bucket"] = pd.qcut(
    df["search_volume"],
    q=4,
    duplicates="drop"
)
volume_audit = (
    df.groupby(
        "volume_bucket",
        observed=True
    )
    .agg(
        n=("content_id", "size"),
        median_search_volume=("search_volume", "median"),
        median_impressions_90d=("impressions_90d", "median"),
        median_clicks_90d=("clicks_90d", "median"),
        median_ctr=("ctr", "median"),
        median_position=("avg_position", "median")
    )
    .reset_index()
)

display(volume_audit)

Zero search-volume rows: 11081


,volume_bucket,n,median_search_volume,median_impressions_90d,median_clicks_90d,median_ctr,median_position
0,"(-0.001, 10.0]",18392,0.0,929.0,1.0,0.09,11.3
1,"(10.0, 20.0]",2290,20.0,1006.5,1.0,0.10,10.8
2,"(20.0, 74000.0]",6850,90.0,842.5,1.0,0.05,12.8


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

In [14]:
df["content_age_bucket"] = pd.qcut(
    df["content_age_days"],
    q=4,
    duplicates="drop"
)

content_age_audit = (
    df.groupby(
        "content_age_bucket",
        observed=True
    )
    .agg(
        n=("content_id", "size"),
        median_impressions=("impressions_90d", "median"),
        median_clicks=("clicks_90d", "median"),
        median_ctr=("ctr", "median")
    )
    .reset_index()
)

display(content_age_audit)

,content_age_bucket,n,median_impressions,median_clicks,median_ctr
0,"(89.999, 132.0]",7518,768.0,1.0,0.12
1,"(132.0, 236.0]",8128,796.5,1.0,0.06
2,"(236.0, 333.0]",6917,601.0,1.0,0.06
3,"(333.0, 564.0]",7437,733.0,1.0,0.05


In [15]:
df["ctr_bucket"] = pd.qcut(
    df["ctr"],
    q=4,
    duplicates="drop"
)

ctr_audit = (
    df.groupby(
        "ctr_bucket",
        observed=True
    )
    .agg(
        n=("content_id", "size"),
        median_impressions=("impressions_90d", "median"),
        median_clicks=("clicks_90d", "median"),
        median_position=("avg_position", "median")
    )
    .reset_index()
)

display(ctr_audit)

,ctr_bucket,n,median_impressions,median_clicks,median_position
0,"(-0.001, 0.07]",15224,129.0,0.0,12.7
1,"(0.07, 0.29]",7503,2960.0,5.0,11.1
2,"(0.29, 100.0]",7273,1890.0,10.0,8.3


In [16]:
df["position_bucket"] = pd.qcut(
    df["avg_position"],
    q=4,
    duplicates="drop"
)

position_audit = (
    df.groupby(
        "position_bucket",
        observed=True
    )
    .agg(
        n=("content_id", "size"),
        median_impressions=("impressions_90d", "median"),
        median_clicks=("clicks_90d", "median"),
        median_ctr=("ctr", "median")
    )
    .reset_index()
)

display(position_audit)

,position_bucket,n,median_impressions,median_clicks,median_ctr
0,"(-0.001, 6.2]",7543,404.0,1.0,0.10
1,"(6.2, 10.8]",7534,982.0,2.0,0.12
2,"(10.8, 22.3]",7462,848.0,1.0,0.10
3,"(22.3, 245.0]",7461,619.0,0.0,0.00


In [13]:
audit_columns = [
    "search_volume",
    "competition",
    "cpc",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

audit_columns = [
    col for col in audit_columns
    if col in df.columns
]

correlation_table = (
    df[audit_columns]
    .corr(numeric_only=True)
)

display(correlation_table)

,search_volume,competition,cpc,impressions_90d,clicks_90d,pageviews_90d,sessions_90d,content_age_days,days_since_last_update,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct
search_volume,1.000000,0.049887,0.042085,0.001203,-0.011260,-0.015152,-0.013171,0.090720,-0.003237,-0.003430,0.045354,-0.010146,-0.013808,-0.003250
competition,0.049887,1.000000,0.302415,-0.053530,-0.032160,-0.071989,-0.063738,0.053409,-0.033763,-0.019170,0.049036,0.000694,-0.038098,0.007956
cpc,0.042085,0.302415,1.000000,-0.025599,-0.023418,-0.028970,-0.029505,0.072067,-0.003417,-0.002252,0.044194,0.004044,-0.018809,-0.002414
impressions_90d,0.001203,-0.053530,-0.025599,1.000000,0.696281,0.582563,0.629925,-0.000743,0.081597,-0.018950,-0.070786,0.024318,-0.096905,-0.008518
clicks_90d,-0.011260,-0.032160,-0.023418,0.696281,1.000000,0.699304,0.765661,-0.022678,0.044444,0.010609,-0.099304,0.030527,-0.062235,-0.010369
pageviews_90d,-0.015152,-0.071989,-0.028970,0.582563,0.699304,1.000000,0.973897,-0.040271,0.186787,-0.004177,-0.017177,0.006455,-0.113303,-0.006670
sessions_90d,-0.013171,-0.063738,-0.029505,0.629925,0.765661,0.973897,1.000000,-0.028436,0.161640,-0.006055,-0.032629,0.005895,-0.110595,-0.010088
content_age_days,0.090720,0.053409,0.072067,-0.000743,-0.022678,-0.040271,-0.028436,1.000000,0.038343,0.009460,0.158238,0.039776,-0.116449,-0.001195
days_since_last_update,-0.003237,-0.033763,-0.003417,0.081597,0.044444,0.186787,0.161640,0.038343,1.000000,-0.020760,0.070140,-0.013661,-0.146887,-0.010358
ctr,-0.003430,-0.019170,-0.002252,-0.018950,0.010609,-0.004177,-0.006055,0.009460,-0.020760,1.000000,-0.072590,0.096903,0.012955,0.005610


In [18]:
missing_audit = pd.DataFrame({
    "column": df.columns,
    "missing_n": [
        df[col].isna().sum()
        for col in df.columns
    ],
    "missing_pct": [
        df[col].isna().mean() * 100
        for col in df.columns
    ]
})

missing_audit = missing_audit.sort_values(
    "missing_pct",
    ascending=False
)

display(missing_audit)

,column,missing_n,missing_pct
10,provider_used,21438,71.460000
9,char_count,7699,25.663333
8,word_count,7699,25.663333
33,word_count_tier,7699,25.663333
34,char_count_tier,7699,25.663333
11,model_used,5733,19.110000
43,trend_pct,3388,11.293333
4,competition_level,2610,8.700000
3,competition,2468,8.226667
45,volume_bucket,2468,8.226667


In [19]:
leakage_keywords = [
    "future",
    "label",
    "target",
    "trend"
]

possible_leakage = [
    col
    for col in df.columns
    if any(
        keyword in col.lower()
        for keyword in leakage_keywords
    )
]

print("Potential leakage-related columns:")
print(possible_leakage)

Potential leakage-related columns:
['trend_direction', 'trend_pct']


In [20]:
signal_summary = pd.DataFrame({
    "signal": [
        "days_since_last_update",
        "search_volume",
        "content_age_days",
        "ctr",
        "avg_position"
    ],
    "role": [
        "Content freshness / staleness",
        "Search demand",
        "Content age",
        "Search-result engagement",
        "Search ranking"
    ],
    "flag_linked": [
        "Yes - Refresh",
        "Yes - Quick Win",
        "Related to freshness",
        "Yes - CTR logic",
        "Yes - CTR/position logic"
    ],
    "verdict": [
        "MIXED",
        "MIXED",
        "Review from audit",
        "Review from audit",
        "Review from audit"
    ]
})

display(signal_summary)

,signal,role,flag_linked,verdict
0,days_since_last_update,Content freshness / staleness,Yes - Refresh,MIXED
1,search_volume,Search demand,Yes - Quick Win,MIXED
2,content_age_days,Content age,Related to freshness,Review from audit
3,ctr,Search-result engagement,Yes - CTR logic,Review from audit
4,avg_position,Search ranking,Yes - CTR/position logic,Review from audit


## Signal Audit Conclusions

The signal audit tested several candidate features for the content-refresh lane.

1. `days_since_last_update` is directly connected to the content-refresh/staleness concept, but its relationship with current performance is mixed across buckets.

2. `search_volume` provides information about potential search demand, but it does not consistently correspond to current page performance across all buckets.

3. `content_age_days`, `ctr`, and `avg_position` provide additional evidence that can be considered when improving the baseline. Their usefulness should be judged from the observed bucket patterns rather than assumed.

4. The audit supports using staleness and search demand as transparent prioritization signals, while recognizing that they are imperfect.

5. Trend-derived fields such as `trend_direction` and `trend_pct` were excluded from the baseline to avoid leakage and future/outcome-derived information.

Overall, the audit shows that the baseline is intentionally simple and has identifiable weaknesses. These weaknesses provide useful directions for the later machine-learning model.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.